<img src="https://ga-curriculum.github.io/universal-resources-internal/ga-logos/ga-red-cog-72x72.png" alt="" style="float: left; margin: 20px 20px 0 0; height: 55px">

# Low Rank Adaptation (LoRA)

---

### Learning Objectives 
- LoRA implementation in Hugging Face  

<a id="home"></a>
### Notebook Guide
- Intro NLP Scenario
- Intro to Hugging Face PEFT
- LoRA demo and comparison of config 

  
### Installs
!pip install -q peft 

[Learn more](https://huggingface.co/docs/peft/main/en/task_guides/lora_based_methods)


### Imports 

In [3]:
from transformers import AutoModelForCausalLM, Trainer, AutoTokenizer, TrainingArguments
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType

### NLP Scenario

You are an analyst for a marketing company that just launched a new product suite of mobile devices. You would like to automate the customer service for this new device. A general LLM hasn't been working well so you are considering fine tuning your model using LoRA to adapt the model to your specific use case. 



### Setup sample data
You have a sample past customer inquiries and expert responses. 

For this demo, you will use only one example. Note you would need several thousand across a diverse set of scenarios to properly fine-tune your model. This would require GPUs. 

In [24]:
customer_service_data = [
    {
        "inquiry": """I'm having trouble with my TechWave X1 battery - it's draining much faster than usual. Just bought it last month and now it barely lasts half a day. What should I do?""",
        
        "response": """I understand your concern about the TechWave X1's battery life. Let's check a few things:

                            1. First, go to Settings > Battery to check which apps are consuming the most power
                            2. Check if any recent apps were installed when the issue started
                            3. Ensure your device has the latest software update
                            4. Try restarting your device to see if that helps  

                    If these steps don't help, please share your battery usage screenshot and I'll help investigate further."""
    }
]



In [5]:
#sets up data using the hugging face Dataset lib
def prepare_dataset(data):
    # Format data for training
    return Dataset.from_dict({
        'inquiry': [x['inquiry'] for x in data],
        'response': [x['response'] for x in data]
    })

# Create training dataset
dataset = prepare_dataset(customer_service_data)

### Load the base model and tokenizer


In [25]:
# Load base model and tokenizer
model_name = "facebook/opt-350m"  # a small-ish model f
base_model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)


def tokenize_function(examples):
    prompt_template = "Customer: {}\n\nAgent: {}"
    texts = [prompt_template.format(i, r) for i, r in 
             zip(examples["inquiry"], examples["response"])]
    
    # First tokenize
    tokenized = tokenizer(texts, padding=True)
    # Then add labels
    tokenized["labels"] = tokenized["input_ids"].copy()
    
    return tokenized

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [26]:
tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

### Define LoRA Config

This is a sample of how to configurate your LoRA 

```python
    lora_config = LoraConfig(
        r=8,                     # Rank of update matrices
        target_modules=["q_proj", "v_proj"],  # Which modules to apply LoRA to
    )

```
where:
- rank: determines LoRA matrix size and typically ranges between 2-32 with 8 being the default
- target_modules:which layers of the base model neural net will be modified. 
    - "q_proj" = Query projection layer
    - "v_proj" = Value projection layer
    - "k_proj" = Key projection layer 


Learn more about the parameters you can adjust here:   
[Hugging Face LoRA Docs](https://huggingface.co/docs/peft/v0.14.0/en/package_reference/lora#peft.LoraConfig)

In [41]:
# Define LoRA Configuration
lora_config = LoraConfig(
    r=8,  # 8 is the default 
    target_modules=["q_proj", "v_proj"],  # Which layers of model to modify 
    task_type="CAUSAL_LM" 
)


#### Get PEFT Model
`get_peft_model` takes two arguments a base model (often loaded from transformers library and your lora config) this sets up for fine-tuning

by calling ```model.print_trainable_parameters()`` you can view details of your model and review is available to be trained

In [42]:
# Get PEFT model
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable params: 1,179,648 || all params: 332,376,064 || trainable%: 0.3549


#### Hugging Face Transformers Training Class
Now you can use this setup with [Hugging Face Trainer class](https://huggingface.co/docs/transformers/en/main_classes/trainer) or [Accelerate](https://huggingface.co/docs/accelerate/en/index) (great for distributed setups) to fine-tune the model 

We will use the [Hugging Face Trainer class](https://huggingface.co/docs/transformers/en/main_classes/trainer) here

We will first setup the arguments we will use for training. 

In [43]:
#traing arguments
training_args = TrainingArguments(
    output_dir="./lora_output",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    warmup_steps=100,
    logging_steps=10,
    save_strategy="epoch",
    learning_rate=1e-4,
)

In [44]:
# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

trainer.train()

Step,Training Loss


TrainOutput(global_step=3, training_loss=1.5520747502644856, metrics={'train_runtime': 17.8923, 'train_samples_per_second': 0.168, 'train_steps_per_second': 0.168, 'total_flos': 1501979443200.0, 'train_loss': 1.5520747502644856, 'epoch': 3.0})

In [29]:
model.save_pretrained("model_output")

### Try It!

1. Create three different LoRA configurations by modifying these key parameters:
    - Config A: original (below)
    - Config B: (lower the rank) from 8 to 4
    - Config C: Add "k_proj" to target_modules

2. Which would run fastest? Which may be more accurate?
   
```python
# Configuration A (Original)
lora_config_a = LoraConfig(
    r=8,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM"
)

# Example for first configuration
model_a = get_peft_model(base_model, lora_config_a)
print("Configuration A:")
model_a.print_trainable_parameters()
```

#### Question 1: Code

In [39]:
# Configuration A (Original)
lora_config_a = LoraConfig(
    r=8,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM"
)

# Example for first configuration
model_a = get_peft_model(base_model, lora_config_a)
print("Configuration A:")
model_a.print_trainable_parameters()

Configuration A:
trainable params: 1,179,648 || all params: 332,376,064 || trainable%: 0.3549


In [40]:
# Configuration B (Lower rank)
lora_config_b = LoraConfig(
    r=4,  # Reduced rank
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM"
)

# Example for second configuration
model_b = get_peft_model(base_model, lora_config_b)
print("Configuration B:")
model_b.print_trainable_parameters()

Configuration B:
trainable params: 786,432 || all params: 331,982,848 || trainable%: 0.2369


In [45]:
# Configuration C (More target modules)
lora_config_c = LoraConfig(
    r=8,
    target_modules=["q_proj", "v_proj", "k_proj"],  # Added k_proj
    task_type="CAUSAL_LM"
)

# Example for final configuration
model_c = get_peft_model(base_model, lora_config_c)
print("Configuration C:")
model_c.print_trainable_parameters()

Configuration C:
trainable params: 1,179,648 || all params: 332,376,064 || trainable%: 0.3549


#### Question 2: Answer

Config B will run the fastest becuase it will have the fewest parameters and smallest matrices, but it may not perform as well as the other models that capture more information